In [10]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import (ChatPromptTemplate,
                                    PromptTemplate,
                                    SystemMessagePromptTemplate,
                                    HumanMessagePromptTemplate)

In [11]:
load_dotenv()

llm =ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen3"
)

In [ ]:
class Joke(BaseModel):
    """Joke to tell user"""
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] = Field(description="The rating of the joke from 1 to 10", default=None)


In [13]:
parser = PydanticOutputParser(pydantic_object=Joke)
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"description": "Joke to tell user", "properties": {"setup": {"description": "The setup of the joke", "title": "Setup", "type": "string"}, "punchline": {"description": "The punchline of the joke", "title": "Punchline", "type": "string"}, "rating": {"anyOf": [{"type": "integer"}, {"type": "null"}], "description": "The rating of the joke from 1 to 10", "title": "Rating"}}, "required": ["setup", "punchline", "rating"]}\n```'

In [21]:
prompt = PromptTemplate(
    template = '''
    Answer the user query with a joke. Here is your formatting instructions: {format_instructions}
    User Query: {query}

    Answer:
    ''',
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = prompt | llm |parser
output = chain.invoke({"query": "Tell me a joke about programming"})

In [20]:
print(output)

setup='Why do programmers prefer dark mode?' punchline='Because the light attracts bugs.' rating=8


In [23]:
from langchain_core.output_parsers import JsonOutputParser
class Movie(BaseModel):
    """Details about a movie"""
    title: str = Field(description = "The title of the movie")
    director: str = Field(description = "The director of the movie")
    year: int = Field(description = "The year the movie was released")
    genre: str = Field(description = "The genre of the movie")

movie_parser = JsonOutputParser(pydantic_object=Movie)
movie_prompt = PromptTemplate(
    template = '''
    Answer the user query with details about a movie. Here is your formatting instructions: {format_instructions}
    User Query: {query} 
    Answer: 
    ''',
    input_variables=["query"],
    partial_variables={"format_instructions": movie_parser.get_format_instructions()}
)
movie_chain = movie_prompt | llm | movie_parser
movie_output = movie_chain.invoke({"query": "Tell me about the movie Inception"})
print(movie_output)

{'title': 'Inception', 'director': 'Christopher Nolan', 'year': 2010, 'genre': 'Science Fiction'}
